In [ ]:
from jointfm_client import bootstrap_notebook
bootstrap_notebook(add_src_root=True)

PosixPath('/home/stefan/workspace/joint-client-python')

# Quantile Forecast
Forecast USD portfolio NAV and risk from 100 positive float daily observations: equity index level, 10-year Treasury yield, and one EUR/USD FX rate. Request quantile forecasts for portfolio NAV and realized volatility over the next 10 ordinal horizons.

In [ ]:
from pathlib import Path

import pandas as pd

from jointfm_client import ColumnSpec, JointFMClient

HISTORY_PATH = Path('notebooks/history.csv')
FEATURE_COLUMNS = ['equity_index_level', 'treasury_10y_yield', 'eur_usd_rate']
TARGET_COLUMNS = ['portfolio_nav', 'realized_volatility']
QUANTILES = [0.1, 0.3, 0.5, 0.7,0.9]
INPUT_STEPS = 100
OUTPUT_HORIZONS = 10
EXPECTED_COLUMNS = FEATURE_COLUMNS + TARGET_COLUMNS
QUERY_TIMES = list(range(INPUT_STEPS, INPUT_STEPS + OUTPUT_HORIZONS))

history = pd.read_csv(HISTORY_PATH, dtype=float)
if list(history.columns) != EXPECTED_COLUMNS:
    raise ValueError(f'Expected columns {EXPECTED_COLUMNS!r}, got {list(history.columns)!r}')
if len(history) != INPUT_STEPS:
    raise ValueError(f'Expected {INPUT_STEPS} history rows, got {len(history)}')

columns = [
    *[
        ColumnSpec(name=column_name, modality='numeric', role='feature')
        for column_name in FEATURE_COLUMNS
    ],
    *[
        ColumnSpec(name=column_name, modality='numeric', role='target')
        for column_name in TARGET_COLUMNS
    ],
]
client = JointFMClient.from_env()
result = client.forecast_quantiles(
    history,
    query_times=QUERY_TIMES,
    requested_columns=TARGET_COLUMNS,
    columns=columns,
    quantiles=QUANTILES,
    seed=7,
)
forecast = result.to_pandas_tidy()
expected_forecast_rows = OUTPUT_HORIZONS * len(TARGET_COLUMNS) * len(QUANTILES)
if len(forecast) != expected_forecast_rows:
    raise ValueError(f'Expected {expected_forecast_rows} forecast rows, got {len(forecast)}')
forecast

,quantile,query_time,requested_column,value
0,0.1,100,portfolio_nav,111.083778
1,0.1,100,realized_volatility,0.010486
2,0.1,101,portfolio_nav,110.698181
3,0.1,101,realized_volatility,0.010343
4,0.1,102,portfolio_nav,111.000069
5,0.1,102,realized_volatility,0.010476
6,0.1,103,portfolio_nav,110.781807
7,0.1,103,realized_volatility,0.010023
8,0.1,104,portfolio_nav,110.523285
9,0.1,104,realized_volatility,0.010560
